# Data Formatting
## customer_id / postal_code の変換テーブル作成
長い hex 文字列を連番整数に変換する。

In [ ]:
%load_ext cudf.pandas

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
print(f"rows: {len(customers):,}  unique customer_id: {customers['customer_id'].nunique():,}")
customers.head(3)

In [ ]:
# customer_id 変換テーブル
unique_ids = customers["customer_id"].unique()
id_map = pd.DataFrame({
    "customer_id": unique_ids,
    "customer_id_int": range(1, len(unique_ids) + 1)
})

id_map.to_csv(PROCESSED_DIR / "customer_id_map.csv", index=False)
print(f"変換テーブル保存完了: {PROCESSED_DIR / 'customer_id_map.csv'}")
id_map.head(3)

In [ ]:
# postal_code 変換テーブル
unique_postal = customers["postal_code"].unique()
postal_map = pd.DataFrame({
    "postal_code": unique_postal,
    "postal_code_int": range(1, len(unique_postal) + 1)
})

postal_map.to_csv(PROCESSED_DIR / "postal_code_map.csv", index=False)
print(f"変換テーブル保存完了: {PROCESSED_DIR / 'postal_code_map.csv'}")
print(f"unique postal_code: {len(postal_map):,}")
postal_map.head(3)

In [ ]:
# customer_id と postal_code を両方整数に変換して保存
customers_converted = (
    customers
    .merge(id_map, on="customer_id", how="left")
    .merge(postal_map, on="postal_code", how="left")
    .drop(columns=["customer_id", "postal_code"])
    .rename(columns={"customer_id_int": "customer_id", "postal_code_int": "postal_code"})
)
# 列順を元に合わせる
original_cols = ["customer_id", "FN", "Active", "club_member_status", "fashion_news_frequency", "age", "postal_code"]
customers_converted = customers_converted[original_cols]

customers_converted.to_csv(PROCESSED_DIR / "customers_converted.csv", index=False)
print(f"変換済み customers 保存完了: {PROCESSED_DIR / 'customers_converted.csv'}")
customers_converted.head(3)